ici on va commencer avec le rapport de notre NLP phase d'avance 

**Énoncé du problème** : les hallucinations se produisent lorsque un grand model de langage geenre des reponses erroné ou qui ne correspondent pas au probleme fournie . et le fait que c'est model sont probabiliste on aura toujours une part d'incertitude et notre but est de determioné quand cela se produit.

**L'architecture hybride de détection** : notre architecture de decison se base sur trpois grands points qui sont :  **Le filtrage (Rule-Based Systems)**,**L'expert sémantique (Fine-tuning)**,**Le débat (Système multi-agent)**


pour evaluer ce model on pourra se baser sur des emtriques comme la f1 score la precison ou le recall 

**travaux connexe**: ici on va parlé des differents approches qui sont generalement utilisé dans cette branche de recherche et faire une critique 
    -il existe differentes approches generalement utilisé comme selfcheckGpt qui bien que performante elle est extremenent couteuse entemps de calcul 
    -le fine tunig classique cette approche n'est pas tres fiable parceuqe le model simple comme le GRU ou le bayesien nayes  peuvent faire du shortcut learning en memorisant les quand l'hallcination se produit sur un dataset en particuler mais ne compreant pas comment faire la classification.

**Experience** : on va parler de ceque nous avons fait 

In [1]:
import json
import random
import csv
import os
from openai import OpenAI

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Initialisation du client OpenAI (assure-toi que OPENAI_API_KEY est dans tes variables d'environnement)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

CHEMIN_DATASET = "dialogue_data.json"
FICHIER_RESULTATS = "resultats_multi_agent.csv"
TAILLE_ECHANTILLON = 10 # Nombre de dialogues à tester (donnera 20 évaluations : 10 vraies, 10 fausses)

# ==========================================
# 2. LE TRIBUNAL MULTI-AGENT
# ==========================================
def appeler_llm(system_prompt, user_prompt):
    """Fonction générique pour interroger l'API OpenAI."""
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo", # Modèle rapide et économique pour les tests
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur API : {e}"

def tribunal_multi_agent(question, reponse_llm, knowledge):
    """Orchestre le débat entre les 3 agents et retourne le verdict et les arguments."""
    contexte_base = f"KNOWLEDGE: {knowledge}\nQUESTION: {question}\nREPONSE A EVALUER: {reponse_llm}"
    
    PROMPT_DEFENSEUR = "Tu es l'Avocat de la Défense. Prouve que la 'REPONSE A EVALUER' est correcte en citant strictement le 'KNOWLEDGE'. Ne cherche pas les erreurs. (3 phrases max)."
    PROMPT_PROCUREUR = "Tu es le Procureur. Prouve que la 'REPONSE A EVALUER' est une hallucination en cherchant des contradictions ou inventions par rapport au 'KNOWLEDGE'. (3 phrases max)."
    PROMPT_JUGE = "Tu es le Juge. Lis le Knowledge, la Réponse et les arguments. Rends ton verdict FINAL sous la forme d'un seul chiffre strict : '1' (Hallucination) ou '0' (Correcte). Ne justifie pas ta réponse, donne juste le chiffre."

    # Tour 1 : Débat
    arg_defense = appeler_llm(PROMPT_DEFENSEUR, contexte_base)
    arg_attaque = appeler_llm(PROMPT_PROCUREUR, contexte_base)
    
    # Tour 2 : Jugement
    dossier_juge = f"{contexte_base}\n\nARGUMENT DEFENSE: {arg_defense}\nARGUMENT ATTAQUE: {arg_attaque}\n\nVerdict (0 ou 1) :"
    verdict_brut = appeler_llm(PROMPT_JUGE, dossier_juge)
    
    # Nettoyage pour s'assurer d'avoir un entier (0 ou 1)
    verdict = 1 if "1" in verdict_brut else 0
    
    return verdict, arg_defense, arg_attaque

# ==========================================
# 3. TRAITEMENT DES DONNÉES ET ÉVALUATION
# ==========================================
def preparer_et_evaluer_dataset():
    print(f"--- 1. Chargement de {TAILLE_ECHANTILLON} dialogues depuis {CHEMIN_DATASET} ---")
    
    toutes_les_lignes = []
    with open(CHEMIN_DATASET, 'r', encoding='utf-8') as f:
        for ligne in f:
            toutes_les_lignes.append(json.loads(ligne))
            
    # Sélection aléatoire de l'échantillon pour avoir des données variées
    echantillon = random.sample(toutes_les_lignes, TAILLE_ECHANTILLON)
    
    # Création du fichier CSV pour sauvegarder les résultats
    with open(FICHIER_RESULTATS, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['ID_Dialogue', 'Type_Test', 'Label_Attendu', 'Prediction_Juge', 'Est_Correct', 'Question', 'Reponse_Evaluee', 'Arg_Defense', 'Arg_Attaque']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        bonnes_predictions = 0
        total_tests = TAILLE_ECHANTILLON * 2 # Car on teste la bonne ET la mauvaise réponse pour chaque dialogue

        print(f"--- 2. Lancement du Tribunal sur {total_tests} instances ---")
        
        for index, data in enumerate(echantillon):
            knowledge = data.get('knowledge', '')
            question = data.get('dialogue_history', '')
            
            # --- TEST A : LA BONNE RÉPONSE (Label 0) ---
            reponse_propre = data.get('right_response', '')
            print(f"Dialogue {index+1}/{TAILLE_ECHANTILLON} - Évaluation Réponse Humaine (Label 0)...")
            pred_0, def_0, att_0 = tribunal_multi_agent(question, reponse_propre, knowledge)
            
            correct_0 = 1 if pred_0 == 0 else 0
            bonnes_predictions += correct_0
            
            writer.writerow({
                'ID_Dialogue': index, 'Type_Test': 'Humaine', 'Label_Attendu': 0, 'Prediction_Juge': pred_0,
                'Est_Correct': correct_0, 'Question': question, 'Reponse_Evaluee': reponse_propre,
                'Arg_Defense': def_0, 'Arg_Attaque': att_0
            })

            # --- TEST B : L'HALLUCINATION (Label 1) ---
            reponse_hallucinee = data.get('hallucinated_response', '')
            print(f"Dialogue {index+1}/{TAILLE_ECHANTILLON} - Évaluation Hallucination (Label 1)...")
            pred_1, def_1, att_1 = tribunal_multi_agent(question, reponse_hallucinee, knowledge)
            
            correct_1 = 1 if pred_1 == 1 else 0
            bonnes_predictions += correct_1
            
            writer.writerow({
                'ID_Dialogue': index, 'Type_Test': 'IA_Hallucinee', 'Label_Attendu': 1, 'Prediction_Juge': pred_1,
                'Est_Correct': correct_1, 'Question': question, 'Reponse_Evaluee': reponse_hallucinee,
                'Arg_Defense': def_1, 'Arg_Attaque': att_1
            })

        # --- RÉSULTATS FINAUX ---
        precision = (bonnes_predictions / total_tests) * 100
        print("\n" + "="*50)
        print(f" ÉVALUATION TERMINÉE !")
        print(f" Précision globale : {precision:.2f}% ({bonnes_predictions}/{total_tests} corrects)")
        print(f" Les résultats détaillés ont été sauvegardés dans : {FICHIER_RESULTATS}")
        print("="*50)

if __name__ == "__main__":
    preparer_et_evaluer_dataset()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable